# 02 — Text Preprocessing and Feature Engineering

This notebook cleans complaint text, adds lightweight extra features, and saves only processed text data. It does **not** save TF-IDF columns into CSV.

In [8]:
# 02_TEXT_PREPROCESSING_AND_FEATURE_ENGINEERING.ipynb
# Customer Complaint Classification System

import os
import re
import pandas as pd
import nltk

from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

# Project paths

BASE_DIR = r"C:\Users\v-amaniyar\Downloads\Adhyayan_presentations\NLP_PROJECT"

OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")
MODEL_DIR = os.path.join(BASE_DIR, "models")
REPORT_DIR = os.path.join(BASE_DIR, "reports")

for folder in [OUTPUT_DIR, MODEL_DIR, REPORT_DIR]:
    os.makedirs(folder, exist_ok=True)

INPUT_FILE = os.path.join(OUTPUT_DIR, "01_nlp_base_complaints.csv")
PROCESSED_FILE = os.path.join(OUTPUT_DIR, "02_processed_complaints.csv")

print("Input file:", INPUT_FILE)
print("Processed output file:", PROCESSED_FILE)


Input file: C:\Users\v-amaniyar\Downloads\Adhyayan_presentations\NLP_PROJECT\outputs\01_nlp_base_complaints.csv
Processed output file: C:\Users\v-amaniyar\Downloads\Adhyayan_presentations\NLP_PROJECT\outputs\02_processed_complaints.csv


In [9]:
# Download NLTK resources

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

# POS tagger name can vary by NLTK version.
# This safe block handles both common versions.
try:
    nltk.download("averaged_perceptron_tagger_eng")
except Exception:
    nltk.download("averaged_perceptron_tagger")


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\v-amaniyar\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\v-amaniyar\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\v-amaniyar\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\v-amaniyar\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [10]:
# Load NLP base dataset

df = pd.read_csv(INPUT_FILE)

print("Loaded shape:", df.shape)
df.head()


Loaded shape: (20937, 2)


,complaint_text,category
0,Good morning my name is XXXX XXXX and I appreciate it if you could help me put a stop to Chase Bank cardmember services. \nIn 2018 I wrote to Chase asking for debt verification and what they sent ...,Debt collection
1,I upgraded my XXXX XXXX card in XX/XX/2018 and was told by the agent who did the upgrade my anniversary date would not change. It turned the agent was giving me the wrong information in order to u...,Credit card or prepaid card
2,"Chase Card was reported on XX/XX/2019. However, fraudulent application have been submitted my identity without my consent to fraudulently obtain services. Do not extend credit without verifying th...","Credit reporting, credit repair services, or other personal consumer reports"
3,"On XX/XX/2018, while trying to book a XXXX XXXX ticket, I came across an offer for {$300.00} to be applied towards the ticket if I applied for a rewards card. I put in my information for the off...","Credit reporting, credit repair services, or other personal consumer reports"
4,my grand son give me check for {$1600.00} i deposit it into my chase account after fund clear my chase bank closed my account never paid me my money they said they need to speek with my grand son ...,Checking or savings account


In [11]:
# Improve / Merge Target Labels


category_mapping = {
    "Credit card": "Credit card or prepaid card",
    "Prepaid card": "Credit card or prepaid card",

    "Bank account or service": "Checking or savings account",

    "Payday loan": "Payday loan, title loan, or personal loan",
    "Consumer Loan": "Vehicle loan or lease",

    "Money transfers": "Money transfer, virtual currency, or money service",
    "Virtual currency": "Money transfer, virtual currency, or money service",

    # Optional: merge old credit reporting name with new one
    "Credit reporting": "Credit reporting, credit repair services, or other personal consumer reports"
}

print("Categories before mapping:", df["category"].nunique())

df["category"] = df["category"].replace(category_mapping)

print("Categories after mapping:", df["category"].nunique())

print("\nCategory distribution after mapping:")
print(df["category"].value_counts())

Categories before mapping: 17
Categories after mapping: 10

Category distribution after mapping:
category
Credit card or prepaid card                                                     7102
Checking or savings account                                                     5936
Mortgage                                                                        3246
Credit reporting, credit repair services, or other personal consumer reports    2037
Debt collection                                                                  930
Money transfer, virtual currency, or money service                               853
Vehicle loan or lease                                                            639
Student loan                                                                     140
Payday loan, title loan, or personal loan                                         45
Other financial service                                                            9
Name: count, dtype: int64


In [19]:
# Helper functions
# Required Imports and NLTK Setup

import re
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer

# Download required NLTK resources
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("averaged_perceptron_tagger")

# Create stopwords and lemmatizer objects
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

print("NLTK setup completed successfully.")
def get_wordnet_pos(word):

    try:
        tag = nltk.pos_tag([word])[0][1][0].upper()
    except LookupError:
        nltk.download("averaged_perceptron_tagger")
        tag = nltk.pos_tag([word])[0][1][0].upper()

    tag_dict = {
        "J": wordnet.ADJ,
        "N": wordnet.NOUN,
        "V": wordnet.VERB,
        "R": wordnet.ADV
    }

    return tag_dict.get(tag, wordnet.NOUN)


def clean_text(text):

    text = str(text).lower()

    # Remove URLs and emails
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)

    # Remove non-alphabet characters
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    words = text.split()

    cleaned_words = []
    for word in words:
        if word not in stop_words and len(word) > 2:
            lemma = lemmatizer.lemmatize(word, get_wordnet_pos(word))
            cleaned_words.append(lemma)

    return " ".join(cleaned_words)


NLTK setup completed successfully.


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\v-amaniyar\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\v-amaniyar\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\v-amaniyar\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\v-amaniyar\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


In [20]:
# Clean rows again for safety

df = df.dropna(subset=["complaint_text", "category"])
df = df.drop_duplicates(subset=["complaint_text", "category"])
df = df.reset_index(drop=True)

print("Shape after safety cleaning:", df.shape)


Shape after safety cleaning: (20936, 4)


In [21]:
# Feature engineering

print("Creating basic text features...")

df["raw_char_length"] = df["complaint_text"].astype(str).apply(len)
df["raw_word_length"] = df["complaint_text"].astype(str).apply(lambda x: len(x.split()))

print("Cleaning complaint text...")
df["clean_text"] = df["complaint_text"].apply(clean_text)

# Cleaned text length
df["clean_word_length"] = df["clean_text"].astype(str).apply(lambda x: len(x.split()))

df.head()


Creating basic text features...
Cleaning complaint text...


,complaint_text,category,raw_char_length,raw_word_length,clean_text,clean_word_length
0,Good morning my name is XXXX XXXX and I appreciate it if you could help me put a stop to Chase Bank cardmember services. \nIn 2018 I wrote to Chase asking for debt verification and what they sent ...,Debt collection,486,92,good morning name xxxx xxxx appreciate could help put stop chase bank cardmember service write chase ask debt verification sent statement acceptable ask bank validate debt instead receive mail eve...,47
1,I upgraded my XXXX XXXX card in XX/XX/2018 and was told by the agent who did the upgrade my anniversary date would not change. It turned the agent was giving me the wrong information in order to u...,Credit card or prepaid card,355,63,upgraded xxxx xxxx card told agent upgrade anniversary date would change turn agent give wrong information order upgrade account xxxx change anniversary date xxxx xxxx without consent xxxx record ...,31
2,"Chase Card was reported on XX/XX/2019. However, fraudulent application have been submitted my identity without my consent to fraudulently obtain services. Do not extend credit without verifying th...","Credit reporting, credit repair services, or other personal consumer reports",224,32,chase card report however fraudulent application submit identity without consent fraudulently obtain service extend credit without verify identity applicant,19
3,"On XX/XX/2018, while trying to book a XXXX XXXX ticket, I came across an offer for {$300.00} to be applied towards the ticket if I applied for a rewards card. I put in my information for the off...","Credit reporting, credit repair services, or other personal consumer reports",1502,269,try book xxxx xxxx ticket come across offer apply towards ticket apply reward card put information offer within less minute notify via screen decision could make immediately contact xxxx refer cha...,119
4,my grand son give me check for {$1600.00} i deposit it into my chase account after fund clear my chase bank closed my account never paid me my money they said they need to speek with my grand son ...,Checking or savings account,477,95,grand son give check deposit chase account fund clear chase bank close account never paid money say need speek grand son check clear money take chase bank refuse pay money grand son call chase tim...,51


In [22]:
# Remove rows where clean text became empty or too small

df = df[df["clean_text"].str.strip() != ""]
df = df[df["clean_word_length"] >= 3]


df = df.reset_index(drop=True)

print("Final processed shape:", df.shape)
df.head()


Final processed shape: (20928, 6)


,complaint_text,category,raw_char_length,raw_word_length,clean_text,clean_word_length
0,Good morning my name is XXXX XXXX and I appreciate it if you could help me put a stop to Chase Bank cardmember services. \nIn 2018 I wrote to Chase asking for debt verification and what they sent ...,Debt collection,486,92,good morning name xxxx xxxx appreciate could help put stop chase bank cardmember service write chase ask debt verification sent statement acceptable ask bank validate debt instead receive mail eve...,47
1,I upgraded my XXXX XXXX card in XX/XX/2018 and was told by the agent who did the upgrade my anniversary date would not change. It turned the agent was giving me the wrong information in order to u...,Credit card or prepaid card,355,63,upgraded xxxx xxxx card told agent upgrade anniversary date would change turn agent give wrong information order upgrade account xxxx change anniversary date xxxx xxxx without consent xxxx record ...,31
2,"Chase Card was reported on XX/XX/2019. However, fraudulent application have been submitted my identity without my consent to fraudulently obtain services. Do not extend credit without verifying th...","Credit reporting, credit repair services, or other personal consumer reports",224,32,chase card report however fraudulent application submit identity without consent fraudulently obtain service extend credit without verify identity applicant,19
3,"On XX/XX/2018, while trying to book a XXXX XXXX ticket, I came across an offer for {$300.00} to be applied towards the ticket if I applied for a rewards card. I put in my information for the off...","Credit reporting, credit repair services, or other personal consumer reports",1502,269,try book xxxx xxxx ticket come across offer apply towards ticket apply reward card put information offer within less minute notify via screen decision could make immediately contact xxxx refer cha...,119
4,my grand son give me check for {$1600.00} i deposit it into my chase account after fund clear my chase bank closed my account never paid me my money they said they need to speek with my grand son ...,Checking or savings account,477,95,grand son give check deposit chase account fund clear chase bank close account never paid money say need speek grand son check clear money take chase bank refuse pay money grand son call chase tim...,51


In [23]:
# Final processed columns
# Important:
# We save clean_text and metadata features only.

final_columns = [
    "complaint_text",
    "clean_text",
    "category",
    "raw_char_length",
    "raw_word_length",
    "clean_word_length"
]

processed_df = df[final_columns].copy()

processed_df.to_csv(PROCESSED_FILE, index=False, encoding="utf-8")

print("Processed file saved at:")
print(PROCESSED_FILE)
print("Saved shape:", processed_df.shape)


Processed file saved at:
C:\Users\v-amaniyar\Downloads\Adhyayan_presentations\NLP_PROJECT\outputs\02_processed_complaints.csv
Saved shape: (20928, 6)


In [24]:
# Quick verification

check_df = pd.read_csv(PROCESSED_FILE)

print("Reloaded shape:", check_df.shape)
check_df.head()


Reloaded shape: (20928, 6)


,complaint_text,clean_text,category,raw_char_length,raw_word_length,clean_word_length
0,Good morning my name is XXXX XXXX and I appreciate it if you could help me put a stop to Chase Bank cardmember services. \nIn 2018 I wrote to Chase asking for debt verification and what they sent ...,good morning name xxxx xxxx appreciate could help put stop chase bank cardmember service write chase ask debt verification sent statement acceptable ask bank validate debt instead receive mail eve...,Debt collection,486,92,47
1,I upgraded my XXXX XXXX card in XX/XX/2018 and was told by the agent who did the upgrade my anniversary date would not change. It turned the agent was giving me the wrong information in order to u...,upgraded xxxx xxxx card told agent upgrade anniversary date would change turn agent give wrong information order upgrade account xxxx change anniversary date xxxx xxxx without consent xxxx record ...,Credit card or prepaid card,355,63,31
2,"Chase Card was reported on XX/XX/2019. However, fraudulent application have been submitted my identity without my consent to fraudulently obtain services. Do not extend credit without verifying th...",chase card report however fraudulent application submit identity without consent fraudulently obtain service extend credit without verify identity applicant,"Credit reporting, credit repair services, or other personal consumer reports",224,32,19
3,"On XX/XX/2018, while trying to book a XXXX XXXX ticket, I came across an offer for {$300.00} to be applied towards the ticket if I applied for a rewards card. I put in my information for the off...",try book xxxx xxxx ticket come across offer apply towards ticket apply reward card put information offer within less minute notify via screen decision could make immediately contact xxxx refer cha...,"Credit reporting, credit repair services, or other personal consumer reports",1502,269,119
4,my grand son give me check for {$1600.00} i deposit it into my chase account after fund clear my chase bank closed my account never paid me my money they said they need to speek with my grand son ...,grand son give check deposit chase account fund clear chase bank close account never paid money say need speek grand son check clear money take chase bank refuse pay money grand son call chase tim...,Checking or savings account,477,95,51
